# 6 · make — structure accuracy against MSA depth

Joins [Open-Athena/helico](https://github.com/Open-Athena/helico) exp14's per-target GDT-TS and
lDDT to [#247](https://github.com/Open-Athena/MarinFold/issues/247)'s published MSA depth, and
aggregates by depth bin.

**Natural monomers only.** #247's feature table covers exactly the 314 natural FoldBench
monomers and not the 19 de novo designs, which is the right cut here rather than a convenience:
an MSA depth for a sequence nobody has ever seen in nature is not the same quantity, and the
designs behave in the opposite direction on every contact-conditioned arm anyway.

The first bin is everything below 100 sequences — the proteins where an alignment-based method
has least to work with, and where an MSA-free predictor should show its advantage if it has one.

CPU only; both sources are read anonymously from public buckets.

In [1]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [2]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "6_msa_depth"
METRICS = ["gdt_ts", "lddt"]
# Bin edges in sequences. The first bin is <100 by request; the rest split the remaining range
# into four bins of comparable occupancy (#247's quartiles over these 314 are 784 / 3,016 /
# 7,414). Edges are inclusive of the lower bound and exclusive of the upper.
DEPTH_EDGES = [0, 100, 1_000, 5_000, 10_000, float("inf")]
DEPTH_LABELS = ["<100", "100–1k", "1k–5k", "5k–10k", "≥10k"]
DEPTH_COLUMN = "msa_depth"
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
# The re-run Helico arm, as in pair 3: exp14 published its arms on #232's sweep checkpoint, and
# #250 re-ran the MarinFold one on step-363000.
EXTRA_ARM = "experiments/exp250_evals_exploration_notebook/data/helico_step363000/per_target.csv"
PARAMETERS = dict(metrics=METRICS, depth_edges=[e if e != float("inf") else None
                                                for e in DEPTH_EDGES],
                  depth_labels=DEPTH_LABELS, depth_column=DEPTH_COLUMN,
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED,
                  extra_arm=EXTRA_ARM)
PARAMETERS

{'metrics': ['gdt_ts', 'lddt'],
 'depth_edges': [0, 100, 1000, 5000, 10000, None],
 'depth_labels': ['<100', '100–1k', '1k–5k', '5k–10k', '≥10k'],
 'depth_column': 'msa_depth',
 'bootstrap_draws': 2000,
 'bootstrap_seed': 0,
 'extra_arm': 'experiments/exp250_evals_exploration_notebook/data/helico_step363000/per_target.csv'}

In [3]:
import numpy as np
import pandas as pd

inputs = figlib.Inputs()
per_target = figlib.load_helico_per_target(inputs, figlib.REPO / EXTRA_ARM,
                                          extra_arm="mf_L_363k")
if per_target.duplicated(["arm", "target_id"]).any():
    raise SystemExit("the pooled table has duplicate (arm, target_id) rows; every arm has "
                     "to appear once per target or the bootstrap below is drawn on "
                     "inflated n")
features = figlib.load_protein_features(inputs)

scored = per_target[per_target.status == "ok"]
# The same rule pair 3 uses: compare arms on one protein set rather than on whichever proteins
# each of them happened to finish.
complete = scored.groupby("target_id").arm.nunique()
keep = set(complete[complete == scored.arm.nunique()].index)
paired = scored[scored.target_id.isin(keep)]

# Joined on `target_id`, not `stem`: the Helico arms carry both, but the baseline rows come
# from their own tables and have a null `stem`, so joining on that silently drops ESMFold,
# ESMFold2 and both Protenix-v2 arms — four of the six predictors this figure is about.
natural = paired[paired.designed == 0].merge(
    features[["stem", DEPTH_COLUMN]].rename(columns={"stem": "target_id"}),
    on="target_id", how="inner", validate="many_to_one")
lost = sorted(set(paired.arm) - set(natural.arm))
if lost:
    raise SystemExit(f"{lost} did not survive the join to #247's features; every arm in the "
                     "paired set has to reach the figure or the comparison is not the one "
                     "the caption claims")
print(f"{natural.arm.nunique()} arms · {paired.target_id.nunique()} paired targets · "
      f"{natural.target_id.nunique()} natural monomers with a published MSA depth")

depths = natural.drop_duplicates("target_id")[DEPTH_COLUMN]
print(f"depth: min {depths.min():,.0f} · median {depths.median():,.0f} · max {depths.max():,.0f}")

note: 333 of 333 rows in per_target.csv are already in the published table and were dropped in its favour; the local copy is now redundant and can be deleted
12 arms · 324 paired targets · 305 natural monomers with a published MSA depth
depth: min 2 · median 3,040 · max 19,393


In [4]:
natural["depth_bin"] = pd.cut(natural[DEPTH_COLUMN], bins=DEPTH_EDGES, labels=DEPTH_LABELS,
                              right=False)
occupancy = natural.drop_duplicates("target_id").depth_bin.value_counts().reindex(DEPTH_LABELS)
print(occupancy.to_string())
if occupancy.min() < 10:
    print(f"note: the smallest bin holds {occupancy.min()} proteins — its interval will be wide")

rows = []
for metric in METRICS:
    for (arm, depth_bin), group in natural.groupby(["arm", "depth_bin"], observed=True):
        values = group[metric].dropna().values
        if not len(values):
            continue
        mean, low, high = figlib.bootstrap_mean(values, BOOTSTRAP_DRAWS, BOOTSTRAP_SEED)
        rows.append(dict(metric=metric, arm=arm, depth_bin=str(depth_bin), n=len(values),
                         value=mean, ci_low=low, ci_high=high))

summary = pd.DataFrame(rows)
summary["depth_bin"] = pd.Categorical(summary.depth_bin, categories=DEPTH_LABELS, ordered=True)
summary = summary.sort_values(["metric", "arm", "depth_bin"])
for metric in METRICS:
    print(f"--- {metric} ---")
    print(summary[summary.metric == metric].pivot(index="arm", columns="depth_bin",
                                                  values="value")
          .to_string(float_format=lambda v: f"{v:.3f}"))

depth_bin
<100       25
100–1k     61
1k–5k     104
5k–10k     66
≥10k       49
--- gdt_ts ---
depth_bin               <100  100–1k  1k–5k  5k–10k  ≥10k
arm                                                      
esmfold                0.423   0.676  0.782   0.862 0.782
esmfold2               0.525   0.756  0.839   0.921 0.839
mf_L                   0.352   0.348  0.482   0.610 0.526
mf_L2                  0.308   0.339  0.440   0.584 0.469
mf_L5                  0.337   0.326  0.363   0.477 0.397
mf_L_363k              0.333   0.396  0.525   0.629 0.559
off                    0.248   0.161  0.136   0.145 0.095
oracle                 0.886   0.898  0.895   0.920 0.852
protenix_v2_msa        0.711   0.860  0.878   0.918 0.872
protenix_v2_single_seq 0.302   0.217  0.153   0.158 0.119
v2msa                  0.692   0.820  0.838   0.875 0.817
v2ss                   0.295   0.215  0.151   0.158 0.115
--- lddt ---
depth_bin               <100  100–1k  1k–5k  5k–10k  ≥10k
arm                   

In [5]:
figlib.write_dataset(
    DATASET,
    notebook="6_make_msa_depth_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_target.csv": lambda path: natural.to_csv(path, index=False),
    },
    extra={
        "protein_class": "natural FoldBench monomers only (#247 publishes no MSA depth for the "
                         "19 de novo designs)",
        "n_proteins": int(natural.target_id.nunique()),
        "occupancy": {label: int(count) for label, count in occupancy.items()},
        "arms": sorted(natural.arm.unique()),
        "depth": {"column": DEPTH_COLUMN, "source": "#247 protein_features.csv",
                  "definition": "sequences in the alignment #247 built for that chain"},
    })

wrote 2 file(s) + metadata.json to /home/bizon/git/MarinFold/.claude/worktrees/evals-exploration-notebook-cba1c7/experiments/exp250_evals_exploration_notebook/figures/data/6_msa_depth
   summary.csv                           9,928 B  263a171329fc
   per_target.csv                      553,976 B  edd215411ca0


PosixPath('/home/bizon/git/MarinFold/.claude/worktrees/evals-exploration-notebook-cba1c7/experiments/exp250_evals_exploration_notebook/figures/data/6_msa_depth')